In [162]:
import pandas as pd
from pathlib import Path
import json

In [183]:
# variables
# l1 = "l1-05-01-2026"
l1= "l1-prompt-v10"
l2 = "l2-05-01-2026"
l3 = "l3-05-01-2026"
l4 = "l4-05-01-2026"
l5 = "l5-05-01-2025"
l6 = "l6-05-01-2025"
eval_run_name= "l1-prompt-v5"

batch = "labelling/artifacts/batches"
outputs = "labelling/artifacts/batch_outputs"
labels = "data/labels"
save_path = f"{labels}/eval/{eval_run_name}/data"


decoder = json.JSONDecoder()

In [184]:
def load_jsonl_folder(folder: str) -> pd.DataFrame:
    """Load and concatenate every .jsonl file in a folder into one DataFrame."""
    folder_path = Path(folder)
    jsonl_paths = sorted(folder_path.glob("*.jsonl"))
    if not jsonl_paths:
        raise FileNotFoundError(f"No .jsonl files found in {folder}")
    frames = [pd.read_json(path, lines=True) for path in jsonl_paths]
    return pd.concat(frames, ignore_index=True)


def load_wos_excels(input_dir: Path) -> pd.DataFrame:
    paths = sorted(
        [path for pattern in ("*.xls", "*.xlsx") for path in Path(input_dir).glob(pattern)]
    )
    if not paths:
        raise FileNotFoundError(f"No .xls or .xlsx files found in {input_dir}")

    frames = [pd.read_excel(path) for path in paths]
    return pd.concat(frames, ignore_index=True)


def extract_output_text(row_obj: dict):
    if not isinstance(row_obj, dict):
        return None

    # row might be the full batch line, the nested response dict, or the body itself
    if "response" in row_obj and isinstance(row_obj["response"], dict):
        body = row_obj["response"].get("body", row_obj["response"])
    else:
        body = row_obj.get("body", row_obj)

    if not isinstance(body, dict):
        return None

    chunks = []
    for item in body.get("output", []):
        if item.get("type") == "message":
            for part in item.get("content", []):
                if part.get("type") == "output_text":
                    chunks.append(part.get("text", ""))

    return "".join(chunks) if chunks else None  


def parse_output_json(text):
    if not isinstance(text, str) or not text.strip():
        return None
    return json.loads(text)


def extract_usage(row_obj: dict):
    """Return token usage fields from a batch row or response/body dict."""
    default = {
        "input_tokens": None,
        "cached_tokens": None,
        "output_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
    }
    if not isinstance(row_obj, dict):
        return default

    if "response" in row_obj and isinstance(row_obj["response"], dict):
        body = row_obj["response"].get("body", row_obj["response"])
    else:
        body = row_obj.get("body", row_obj)

    if not isinstance(body, dict):
        return default

    usage = body.get("usage", {}) or {}
    return {
        "input_tokens": usage.get("input_tokens"),
        "cached_tokens": (usage.get("input_tokens_details") or {}).get("cached_tokens"),
        "output_tokens": usage.get("output_tokens"),
        "reasoning_tokens": (usage.get("output_tokens_details") or {}).get("reasoning_tokens"),
        "total_tokens": usage.get("total_tokens"),
    }


### L1) Drivers

In [185]:
l1_batch_data = load_jsonl_folder(f"{batch}/{l1}/data")
l1_outputs = load_jsonl_folder(f"{outputs}/{l1}")
l1_labels= load_wos_excels(f"{labels}/l1")

In [186]:
l1_outputs["response_json"] = l1_outputs["response"].apply(lambda v: v if isinstance(v, dict) else json.loads(v))
l1_outputs["output_text"] = l1_outputs["response_json"].apply(extract_output_text)
l1_outputs[["input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]] = (
    l1_outputs["response_json"]
    .apply(extract_usage)   # returns a dict per row
    .apply(pd.Series)       # turn each dict into a row of columns
)

l1_outputs["results"] = l1_outputs["output_text"].apply(
    lambda t: (
        (lambda obj: obj.get("results") if isinstance(obj, dict) else None)(
            decoder.raw_decode(t)[0]
        )
        if isinstance(t, str) and t.strip() else None
    )
)

In [187]:
_output_cols = ["Article Title", "Abstract", "DOI", "source", "driver"]

l1_df = l1_outputs.merge(
    l1_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *_output_cols]],
    on="custom_id",
    how="left",
)


### L2) threats

In [82]:
# L2: threats
l2_batch_data = load_jsonl_folder(f"{batch}/{l2}/data")
l2_outputs = load_jsonl_folder(f"{outputs}/{l2}")
l2_labels = load_wos_excels(f"{labels}/l2")

def extract_l2_fields(row):
    """Safely pull threat results and usage across nested responses."""
    def safe_dict(val):
        return val if isinstance(val, dict) else {}

    payload = safe_dict(row.get("results_payload"))
    responses = safe_dict(row.get("responses"))

    out = {
        "threat_l0_results": safe_dict(payload.get("threat_l0")).get("results"),
        "threat_l1_results": safe_dict(payload.get("threat_l1")).get("results"),
        "threat_l2_results": safe_dict(payload.get("threat_l2")).get("results"),
    }

    for key in ("threat_l0", "threat_l1", "threat_l2"):
        usage = safe_dict(safe_dict(responses.get(key)).get("body")).get("usage", {})
        usage = usage if isinstance(usage, dict) else {}
        out.update({
            f"{key}_input_tokens": usage.get("input_tokens"),
            f"{key}_output_tokens": usage.get("output_tokens"),
            f"{key}_total_tokens": usage.get("total_tokens"),
        })

    return pd.Series(out)

l2_extracted = l2_outputs.apply(extract_l2_fields, axis=1)

l2_df = pd.concat([l2_outputs, l2_extracted], axis=1)
l2_label_cols = ["Article Title", "Abstract", "DOI", "source", "threats_l0", "threats_l1"]
l2_df = l2_df.merge(
    l2_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *l2_label_cols]],
    on="custom_id",
    how="left",
)


### L3) geography

In [83]:
# L3: geography
l3_batch_data = load_jsonl_folder(f"{batch}/{l3}/data")
l3_outputs = load_jsonl_folder(f"{outputs}/{l3}")
l3_labels = load_wos_excels(f"{labels}/l3")

l3_outputs["response_json"] = l3_outputs["response"].apply(lambda v: v if isinstance(v, dict) else json.loads(v))
l3_outputs["output_text"] = l3_outputs["response_json"].apply(extract_output_text)
l3_outputs[["input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]] = (
    l3_outputs["response_json"]
    .apply(extract_usage)
    .apply(pd.Series)
)

def safe_geo_json(t):
    try:
        return decoder.raw_decode(t)[0] if isinstance(t, str) and t.strip() else None
    except json.JSONDecodeError:
        return None

l3_outputs["geo_json"] = l3_outputs["output_text"].apply(safe_geo_json)
l3_outputs["regions"] = l3_outputs["geo_json"].apply(lambda d: d.get("regions") if isinstance(d, dict) else None)
l3_outputs["subregions"] = l3_outputs["geo_json"].apply(lambda d: d.get("subregions") or d.get("sub_regions") if isinstance(d, dict) else None)
l3_outputs["countries_iso3"] = l3_outputs["geo_json"].apply(lambda d: d.get("countries_iso3") if isinstance(d, dict) else None)
l3_outputs["locales"] = l3_outputs["geo_json"].apply(lambda d: d.get("locales") if isinstance(d, dict) else None)
l3_outputs["locale_coordinates"] = l3_outputs["geo_json"].apply(lambda d: d.get("locale_coordinates") if isinstance(d, dict) else None)
l3_outputs["locale_lat"] = l3_outputs["locale_coordinates"].apply(lambda coords: [c.get("lat") for c in coords if isinstance(c, dict) and c.get("lat") is not None] if isinstance(coords, list) else None)
l3_outputs["locale_lon"] = l3_outputs["locale_coordinates"].apply(lambda coords: [c.get("lon") for c in coords if isinstance(c, dict) and c.get("lon") is not None] if isinstance(coords, list) else None)

l3_label_cols = ["Article Title", "Abstract", "DOI", "source", "region", "sub-region", "country"]
l3_df = l3_outputs.merge(
    l3_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *l3_label_cols]],
    on="custom_id",
    how="left",
)


### L4) ecosystems

In [84]:
# L4: ecosystems
l4_batch_data = load_jsonl_folder(f"{batch}/{l4}/data")
l4_outputs = load_jsonl_folder(f"{outputs}/{l4}")
l4_labels = load_wos_excels(f"{labels}/l4")

def extract_l4_fields(row):
    """Safely pull realm/biome/efg results and token usage from nested responses."""
    defaults = {
        "realm_results": None,
        "biome_results": None,
        "efg_results": None,
        "realm_input_tokens": None,
        "realm_output_tokens": None,
        "realm_total_tokens": None,
        "biome_input_tokens": None,
        "biome_output_tokens": None,
        "biome_total_tokens": None,
        "efg_input_tokens": None,
        "efg_output_tokens": None,
        "efg_total_tokens": None,
    }

    try:
        def safe_dict(val):
            return val if isinstance(val, dict) else {}

        payload = safe_dict(row.get("results_payload"))
        responses = safe_dict(row.get("responses"))

        out = defaults.copy()
        out["realm_results"] = safe_dict(payload.get("realms")).get("results")
        out["biome_results"] = safe_dict(payload.get("biomes")).get("results")
        out["efg_results"] = safe_dict(payload.get("efgs")).get("results")

        for key in ("realm", "biome", "efg"):
            resp = safe_dict(responses.get(key))
            body = safe_dict(resp.get("body"))
            usage = safe_dict(body.get("usage"))
            out.update({
                f"{key}_input_tokens": usage.get("input_tokens"),
                f"{key}_output_tokens": usage.get("output_tokens"),
                f"{key}_total_tokens": usage.get("total_tokens"),
            })

        return pd.Series(out)
    except Exception:
        return pd.Series(defaults)

l4_extracted = l4_outputs.apply(extract_l4_fields, axis=1)
l4_df = pd.concat([l4_outputs, l4_extracted], axis=1)
l4_label_cols = ["Article Title", "Abstract", "DOI", "source", "realm", "biome"]
l4_df = l4_df.merge(
    l4_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *l4_label_cols]],
    on="custom_id",
    how="left",
)


### L6) Taxa

In [85]:
# L6: taxa
l6_batch_dir = Path(f"{batch}/{l6}/data")
l6_batch_data = load_jsonl_folder(l6_batch_dir) if l6_batch_dir.exists() else None
l6_outputs = load_jsonl_folder(f"{outputs}/{l6}")
l6_labels = load_wos_excels(f"{labels}/l6")

l6_outputs["response_json"] = l6_outputs["response"].apply(lambda v: v if isinstance(v, dict) else json.loads(v))
l6_outputs["output_text"] = l6_outputs["response_json"].apply(extract_output_text)
l6_outputs[["input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]] = (
    l6_outputs["response_json"]
    .apply(extract_usage)
    .apply(pd.Series)
)

def safe_l6_results(t):
    try:
        obj = decoder.raw_decode(t)[0] if isinstance(t, str) and t.strip() else None
    except json.JSONDecodeError:
        obj = None
    return obj.get("results") if isinstance(obj, dict) else None

l6_outputs["results"] = l6_outputs["output_text"].apply(safe_l6_results)

l6_label_cols = ["Article Title", "Abstract", "DOI", "source", "kingdom", "phylum", "class", "order", "specie"]
l6_df = l6_outputs.merge(
    l6_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *l6_label_cols]],
    on="custom_id",
    how="left",
)


### L5) Study Design

In [86]:
# L5: study design
l5_batch_data = load_jsonl_folder(f"{batch}/{l5}/data")
l5_outputs = load_jsonl_folder(f"{outputs}/l5-05-01-2026")
l5_labels = load_wos_excels(f"{labels}/l5")

l5_outputs["response_json"] = l5_outputs["response"].apply(lambda v: v if isinstance(v, dict) else json.loads(v))
l5_outputs["output_text"] = l5_outputs["response_json"].apply(extract_output_text)
l5_outputs[["input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]] = (
    l5_outputs["response_json"]
    .apply(extract_usage)
    .apply(pd.Series)
)

def safe_l5_json(t):
    try:
        return decoder.raw_decode(t)[0] if isinstance(t, str) and t.strip() else None
    except json.JSONDecodeError:
        return None

l5_outputs["study_json"] = l5_outputs["output_text"].apply(safe_l5_json)
l5_outputs["results"] = l5_outputs["study_json"].apply(lambda d: d.get("results") if isinstance(d, dict) else None)

def first_result_field(results, key):
    if isinstance(results, list) and results and isinstance(results[0], dict):
        return results[0].get(key)
    return None

l5_outputs["pred_study_design"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "study_design"))
l5_outputs["pred_methods_data_collection"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "methods_data_collection"))
l5_outputs["pred_methods_analysis"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "methods_analysis"))
l5_outputs["pred_has_comparison"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "has_comparison"))
l5_outputs["pred_comparison_types"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "comparison_types"))
l5_outputs["pred_impact_assessments"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "impact_assessments"))
l5_outputs["pred_trait"] = l5_outputs["results"].apply(lambda r: first_result_field(r, "trait"))

l5_label_cols = ["Article Title", "Abstract", "DOI", "source", "study_design"]
l5_df = l5_outputs.merge(
    l5_labels.rename(columns={"UT (Unique WOS ID)": "custom_id"})[["custom_id", *l5_label_cols]],
    on="custom_id",
    how="left",
)


### Export

In [132]:
# Export per-level DataFrames
# Define these before running: save_path = "outputs"; eval_run_name = "run-001"
export_root = Path(save_path)
export_root.mkdir(parents=True, exist_ok=True)

def export_df(df, level_name):
    folder = export_root / level_name
    folder.mkdir(parents=True, exist_ok=True)
    out_path = folder / f"{level_name}_df.xlsx"
    df.to_excel(out_path, index=False)
    return out_path

exported = {}
for name in ("l1"): # , "l2", "l3", "l4", "l5", "l6"
    df_var = f"{name}_df"
    if df_var in globals():
        path = export_df(globals()[df_var], name)
        exported[name] = path

exported


{}

In [188]:
l1_df.to_excel("data/labels/eval/l1-prompt-v10/data/l1/l1_df.xlsx", index=False)